# NB 01 · Hello LangGraph

**Series:** Learning LangGraph through literate notebooks, notebook 1 of 7.
**Central question:** what changes when the same step is built as a graph instead of a chain?

Over seven notebooks, one research assistant is built that answers questions from a small
collection of fictional finance documents. This first notebook strips the task to its core:
one question, one document fixed in the code, one answer. That step is built twice, once as a
LangChain chain and once as a LangGraph graph, and the two are compared. Retrieval, scoring and
loops come later.

## Learning goals

After this notebook you can:

- write a prompt template and fill it with variables;
- join prompt, model and parser into a chain with the pipe operator `|`;
- build the same step as a `StateGraph` with `add_node`, `add_edge`, `compile()` and `invoke()`;
- explain what a graph's state is and how a node changes it;
- recognise three typical beginner mistakes from their error messages.

## Setup

The first cell loads settings, checks versions and creates the language model. It must run on a
fresh environment; installation is described in the README.

`LLM_MODE` decides which model answers:

- `fake` (default): a scripted stand-in that returns recorded answers. Every run gives the same
  result, with no internet connection and no API key.
- `ollama`: the real local model `qwen3:8b`.
- `disabled`: no model. This notebook needs a model to answer, so it falls back to the scripted one.

You can set `LLM_MODE` in a `.env` file in the project folder; `load_dotenv()` reads it.

In [ ]:
import sys
from importlib.metadata import version
from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, START, StateGraph

from research_assistant.models import get_chat_model, get_llm_mode

load_dotenv()

assert sys.version_info >= (3, 12), "This series needs Python 3.12 or newer."
print(f"Python {sys.version.split()[0]}")
for package in ("langgraph", "langchain-core"):
    print(f"{package} {version(package)}")

mode = get_llm_mode()
if mode == "disabled":
    mode = "fake"  # this notebook needs a model; the scripted one stands in
llm = get_chat_model(mode, fixture="nb01_hello.yaml")
print(f"Model mode: {mode}")

## Concept: a chain and a graph

A **chain** is a fixed sequence. LangChain's expression language (LCEL) joins steps with the pipe
operator `|`, much like a pipeline in a terminal: the output of each step becomes the input of the next.

```text
{question, document} -> prompt -> model -> parser -> answer
```

A **graph** describes the same work in three parts:

- a **state**: a small dictionary with fixed fields that every step can read;
- **nodes**: ordinary Python functions that read the state and return the fields they change;
- **edges**: the connections that decide which node runs next, from the special `START` to the special `END`.

```text
START --> [answer] --> END

state before the run:  {question}
state after the run:   {question, draft_answer}
```

For one straight step, both do the same job. The difference that matters later is that edges in a
graph can branch and lead back to an earlier node, and the state keeps everything the steps have
produced, while a chain only passes its output forward. This notebook builds both versions side by
side before that difference appears.

## Experiment

### 1 The task

One fictional document, written straight into the code, and one question about it.

In [ ]:
DOCUMENT = (
    "Solvane Dynamics AG reported revenue of CHF 342 million for the third quarter "
    "of 2025, an increase of 14.8% compared with the same quarter of the previous year."
)
QUESTION = "What revenue did Solvane Dynamics report for the third quarter of 2025?"

### 2 The prompt

A prompt template is text with placeholders in curly brackets. `ChatPromptTemplate` fills them in
and produces the messages the model receives. This is the final version of the prompt; the prompt
log further down shows the first version and why it was changed.

In [ ]:
NOT_IN_DOCUMENT = "The document does not say."

ANSWER_PROMPT = ChatPromptTemplate.from_template(
    "Answer the question in one sentence, using only the document below. "
    "If the document does not contain the answer, reply exactly: "
    '"' + NOT_IN_DOCUMENT + '"'
    "\n\nDocument: {document}\n\nQuestion: {question}"
)

print(ANSWER_PROMPT.invoke({"document": DOCUMENT, "question": QUESTION}).to_string())

### 3 Version A: the chain

The pipe operator joins three steps: the template fills in the variables, the model answers, and
`StrOutputParser` turns the model's message object into plain text. `invoke()` runs the whole chain
with one dictionary of inputs.

In [ ]:
chain = ANSWER_PROMPT | llm | StrOutputParser()

answer_from_chain = chain.invoke({"document": DOCUMENT, "question": QUESTION})
print(answer_from_chain)

### 4 Version B: the graph

Building a graph takes four steps, each visible in the cell below:

1. **Define the state.** `HelloState` lists the fields the graph carries. `total=False` means a
   field may be missing at the start; the caller only provides `question`.
2. **Write the node.** `answer` is a plain function. It reads `state["question"]`, calls the model
   with the same prompt as the chain, and returns **only the field it changes**:
   `{"draft_answer": ...}`. LangGraph merges this partial update into the state.
3. **Connect the edges.** From `START` to `answer`, then from `answer` to `END`.
4. **Compile.** `compile()` checks the structure and returns a graph that can run.

In [ ]:
class HelloState(TypedDict, total=False):
    question: str
    draft_answer: str


def answer(state: HelloState) -> dict:
    messages = ANSWER_PROMPT.invoke({"document": DOCUMENT, "question": state["question"]})
    reply = llm.invoke(messages)
    return {"draft_answer": reply.content}


builder = StateGraph(HelloState)
builder.add_node("answer", answer)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile()

LangGraph can describe a compiled graph as a Mermaid diagram. The text below can be pasted into any
Mermaid viewer; it shows the same three boxes as the sketch in the concept section.

In [ ]:
print(graph.get_graph().draw_mermaid())

### 5 Run the graph

`invoke()` takes the starting state and returns the final state. The answer must be identical to the
chain's, because both use the same prompt and the same model.

In [ ]:
final_state = graph.invoke({"question": QUESTION})
print(final_state)

assert final_state["draft_answer"] == answer_from_chain

### 6 Watching the graph work

`stream()` runs the graph step by step and reports each node's update as it happens. With one node
there is one update. From NB 03 on, this is how the assistant is followed through retrieval, scoring
and a second search round.

In [ ]:
for update in graph.stream({"question": QUESTION}):
    print(update)

### 7 A question the document cannot answer

The document gives revenue, not net profit. A good answer admits that, in a form that a later step
can recognise.

In [ ]:
UNANSWERABLE_QUESTION = "What net profit did Solvane Dynamics report for the third quarter of 2025?"

result = graph.invoke({"question": UNANSWERABLE_QUESTION})
print(result["draft_answer"])
assert result["draft_answer"] == NOT_IN_DOCUMENT

### 8 So why build a graph at all?

For this task, the chain is shorter and does the same job. A straight line of steps is exactly what
chains are made for. The graph earns its extra lines once the flow stops being a straight line:

- when several steps must share and extend one record of what has been found (NB 02);
- when the next step depends on what an earlier step found, for example searching again because the
  evidence is too thin (NB 04). A chain has no way back to an earlier step; a graph needs only one
  more edge.

Keep this one-node graph in mind: every later notebook adds nodes and edges to the same structure.

## Common mistakes

Each mistake below runs on purpose and fails. The test suite checks that it fails with exactly the
error named here, and with no other.

### Mistake 1: no edge from `START`

A graph needs to know where to begin. Forgetting the first edge is easy when `answer` feels like the
obvious first step.

In [ ]:
broken = StateGraph(HelloState)
broken.add_node("answer", answer)
broken.add_edge("answer", END)
broken.compile()

`compile()` stops with *"Graph must have an entrypoint"*. The check happens while the graph is being
built, before anything runs. **Fix:** add `builder.add_edge(START, "answer")`, as in version B.

### Mistake 2: a node returns the answer itself

Coming from chains, it feels natural for a step to return its result. A node must instead return a
dictionary of the state fields it changes.

In [ ]:
def answer_returning_text(state: HelloState) -> str:
    messages = ANSWER_PROMPT.invoke({"document": DOCUMENT, "question": state["question"]})
    return llm.invoke(messages).content


broken = StateGraph(HelloState)
broken.add_node("answer", answer_returning_text)
broken.add_edge(START, "answer")
broken.add_edge("answer", END)
broken.compile().invoke({"question": QUESTION})

This time `compile()` succeeds, because the structure is fine. The error appears only when the graph
runs: *"Expected dict, got Solvane Dynamics reported..."*. LangGraph cannot know which state field
the text belongs to. **Fix:** return `{"draft_answer": ...}`.

### Mistake 3: a prompt variable is never supplied

The template expects `{document}` and `{question}`. Calling the chain with only the question leaves
one placeholder without a value.

In [ ]:
chain.invoke({"question": QUESTION})

The error names the missing variable and lists what was expected and what was received. **Fix:**
pass every variable the template uses, as in version A. In the graph version this mistake cannot
happen in the same way, because the node builds the inputs itself from the state and the fixed
document.

## Prompt log

The scripted answers in this notebook were recorded from `qwen3:8b` (temperature 0) on
4 October 2026, so this log describes real model behaviour.

**Version 1**

```text
Answer the question.

Document: {document}

Question: {question}
```

**What happened.** For the revenue question, the answer was correct and short. For the net profit
question, which the document cannot answer, the model did not invent a figure. It replied with three
sentences about what the document does and does not contain (the next cell shows the recorded
reply). The content is fine, but the form is not: the wording changes from question to question, so
no later step can reliably tell "no answer found" apart from an answer.

**Version 2 (final)**

```text
Answer the question in one sentence, using only the document below.
If the document does not contain the answer, reply exactly:
"The document does not say."

Document: {document}

Question: {question}
```

**What changed.** The revenue question gets the same one-sentence answer. The net profit question
gets exactly *"The document does not say."*, which code can check with `==`, as the experiment does.

**Insight.** A prompt is an interface. Fixing the output format, and above all the reply for missing
information, matters as much as the instruction itself. Two side observations:

- The first prompt was expected to make the model invent a net profit. With this document and this
  model it did not; the weakness was the form, not the facts. NB 05 returns to the question of when
  models do misjudge evidence.
- By default, `qwen3:8b` "thinks" before it answers. That took about 60 seconds per answer instead of
  about 5, with the same result. The project therefore switches thinking off
  (`OLLAMA_REASONING=false`).

In [ ]:
FIRST_PROMPT = ChatPromptTemplate.from_template(
    "Answer the question.\n\nDocument: {document}\n\nQuestion: {question}"
)
first_chain = FIRST_PROMPT | llm | StrOutputParser()
print(first_chain.invoke({"document": DOCUMENT, "question": UNANSWERABLE_QUESTION}))

With `LLM_MODE=ollama`, the next cell sends both prompt versions and both questions to the live
model. It is skipped in scripted mode and in the automated tests. Live answers can differ slightly
from the recorded ones; that variation is the reason the tests never rely on them.

In [ ]:
for name, prompt in (("Version 1", FIRST_PROMPT), ("Version 2", ANSWER_PROMPT)):
    for question in (QUESTION, UNANSWERABLE_QUESTION):
        reply = (prompt | llm | StrOutputParser()).invoke(
            {"document": DOCUMENT, "question": question}
        )
        print(f"{name} | {question}\n{reply}\n")

## Summary

One step was built twice. The chain joins prompt, model and parser in a fixed line. The graph puts
the same work into a node, connects it with edges and carries the result in an explicit state. Same
prompt, same model, same answer.

The code of this notebook also lives in the package `research_assistant`, in `hello.py`, where it is
tested. Later notebooks import it from there instead of copying it. The next cell checks that the
notebook's version and the package's version still agree. The comparison of a complete run only
happens in scripted mode: a live model can word the same answer slightly differently from one call
to the next, even at temperature 0.

In [ ]:
from research_assistant import hello

assert hello.DOCUMENT == DOCUMENT
assert hello.ANSWER_PROMPT == ANSWER_PROMPT
if mode == "fake":
    assert hello.build_graph(llm).invoke({"question": QUESTION}) == final_state
print("Notebook and package agree.")

**Next: NB 02, the state contract.** Here the state had two fields and a single node writing to it.
As soon as several nodes write to the same state, two questions appear: who may change which field,
and what happens when two nodes write to the same field at the same time? NB 02 answers both.

## Exercise

Ask the graph a third question about the document: *"By what percentage did Solvane Dynamics'
revenue grow compared with the previous year?"*

1. With `LLM_MODE=ollama`, run it and check that the answer is one sentence that uses only the
   document.
2. With `LLM_MODE=fake`, the same call fails with *"no fixture rule matches prompt"*. Explain why,
   then add a rule to `fixtures/fake_llm/nb01_hello.yaml` that makes it pass. What does this tell you
   about what the scripted model can and cannot test?